# `01-scraping.ipynb`: Scraping logs from UMPD website.

## Scraping methodology

UMD's website fortunately uses HTML tables to format its log data, so I will use BeautifulSoup to scrape the tables and load them as pandas dataframes. 

### Scraping methodology for Arrests Log (per year)

### Scraping methodology for Incident and Arrests Logs (per month per year)

In [1]:
# setup 
import os
import requests
import pandas as pd
import geopandas as gpd
import bs4 as bs
import time
import shutil
from pathlib import Path

DATA_DIR = Path.cwd().parent / "data"

HEADERS = {
  "User-Agent": "Jour720/1.0 (diegojto@umd.edu)",
  "Accept-Language": "en-US,en;q=0.9",
}
BASE_ARREST_URL = "https://umpd.umd.edu/statistics-reports/arrest-report-ledgers/"
BASE_CI_URL = "https://umpd.umd.edu/statistics-reports/daily-crime-and-incident-logs/" # CI = Crime and Incident
BASE_UCR_URL = "https://umpd.umd.edu/statistics-reports/uniform-crime-reports"
CLERYACT_URL = "https://umpd.umd.edu/statistics-reports/clery-act-info-reports/clery-act-statistics"


MONTHS = [f"0{i}" for i in range(1, 10)] + [str(i) for i in range(10, 13)]
CURRENT_YEAR = time.localtime().tm_year
CURRENT_MONTH = time.localtime().tm_mon
ARREST_YEARS = list(range(2018, CURRENT_YEAR + 1))
CI_YEARS = list(range(2019, CURRENT_YEAR + 1))

def text(i, cells):
    return cells[i].get_text(strip=True).strip()

## Scraping Arrests Logs

All of the yearly Arrest Ledger links listed on UMPD's [Arrest Logs](https://umpd.umd.edu/statistics-reports/arrest-logs) website have the following URL structure: `https://umpd.umd.edu/statistics-reports/arrest-report-ledgers/YYYY`, with year options in the range between 2018-2026.

In [8]:
# Arrest Ledger URL list build
arrest_data = {}
arrest_urls = [f"{BASE_ARREST_URL}{year}" for year in ARREST_YEARS]
start_time = time.time()
for year, url in zip(ARREST_YEARS, arrest_urls):
    year_start_time = time.time()
    response = requests.get(url, headers=HEADERS, timeout=10)
    response.raise_for_status()  # Raise an error if the request was unsuccessful
    soup = bs.BeautifulSoup(response.text, "html.parser")
    table = soup.find("table")
    rows = table.find_all("tr")[1:] if table else []
    
    assert rows, f"No rows found for year {year} at URL {url}"
    year_data = []
    for row in rows: 
      cells = row.find_all("td")
      
      if len(cells) < 7:
        print(f"Row has insufficient cells: {len(cells)}")
        continue
      
      year_data.append({
        "arrest_number": text(0, cells),
        "arrest_date": text(1, cells),
        "umpd_case_number": text(2, cells),
        "age": text(3, cells),
        "race": text(4, cells),
        'sex': text(5, cells),
        'arrested_charge': text(6, cells)
      })
    year_end_time = time.time()
    print(f"Scraped {len(year_data)} rows for year {year} in {year_end_time - year_start_time:.2f} seconds.")
    arrest_data[year] = year_data
end_time = time.time()
print(f"Total arrest ledger scraping time: {end_time - start_time:.2f} seconds.")
arrest_dfs = {year: pd.DataFrame(data) for year, data in arrest_data.items()}

Scraped 375 rows for year 2018 in 3.22 seconds.
Scraped 685 rows for year 2019 in 5.90 seconds.
Scraped 324 rows for year 2020 in 2.92 seconds.
Scraped 349 rows for year 2021 in 3.38 seconds.
Scraped 357 rows for year 2022 in 3.16 seconds.
Scraped 300 rows for year 2023 in 2.76 seconds.
Scraped 435 rows for year 2024 in 4.13 seconds.
Scraped 440 rows for year 2025 in 4.38 seconds.
Scraped 466 rows for year 2026 in 0.12 seconds.
Total arrest ledger scraping time: 29.96 seconds.


In [9]:
for year, df in arrest_dfs.items():
  df.to_csv(f"../data/raw/arrest-ledger/arrest_ledger_{year}.csv", index=False)
  
arrest_df_all = pd.concat(arrest_dfs.values(), ignore_index=True)
arrest_df_all.to_csv(f"../data/raw/arrest-ledger/umpd_arrest_ledger_2018-{CURRENT_YEAR}.csv", index=False)

## Scraping Incident and Arrests Logs

The [Daily Crime and Incident Logs](https://umpd.umd.edu/statistics-reports/daily-crime-and-incident-logs) have the following URL structure: `https://umpd.umd.edu/statistics-reports/daily-crime-and-incident-logs/YYYY/MM`, year options being (2019-2026) and month options being (01-12), except 2026 will have to have current month.

**NOTE**: The website as of 2026-11-06 has not been updated to include November Daily logs. This code is written to scrape every crime and incident log to the month prior of running this script. 

In [10]:
ci_data = {year: {int(month): [] for month in MONTHS} for year in CI_YEARS}
ci_urls = [f"{BASE_CI_URL}{year}/{month}" for year in CI_YEARS for month in MONTHS if not (year == CURRENT_YEAR and int(month) >= CURRENT_MONTH)]
for url in ci_urls: 
  url_parts = url.split("/")
  year, month = url_parts[-2], url_parts[-1]
  response = requests.get(url, headers=HEADERS, timeout=10)
  response.raise_for_status() 
  soup = bs.BeautifulSoup(response.text, "html.parser")
  table = soup.find("table")
  rows = table.find_all("tr")[1:] if table else []
  
  assert rows, f"No rows found for year {year}, month {month} at URL {url}"
  
  for i in range(len(rows)-1):
    row = rows[i]
    row_next = rows[i+1]
    
    row_cells = row.find_all("td")
    row_next_cells = row_next.find_all("td")
    
    if len(row_cells) == 1 and len(row_next_cells) != 1:
      continue
    
    # if (len(row_cells) < 5 or len(row_next_cells) != 1) and (len(row_cells) + len(row_next_cells) != 6):
    #   print(f"Row {i} or Row {i+1} has insufficient cells: {len(row_cells)} and {len(row_next_cells)}")
    #   continue
    
    ci_data[int(year)][int(month)].append({
      'umpd_case_number': text(0, row_cells),
      'date_occurred': text(1, row_cells),
      'report_date': text(2, row_cells),
      'crime_type': text(3, row_cells),
      'disposition': text(4, row_cells),
      'location': text(0, row_next_cells)
    })


In [11]:
ci_dfs_individual = {year: [] for year in CI_YEARS}
ci_dfs_combined_yearly = {year: None for year in CI_YEARS}
for year, months in ci_data.items():
  os.makedirs(f"../data/raw/daily-crime-incident-logs/{year}", exist_ok=True)
  month_list = []
  for month, data in months.items():
    df = pd.DataFrame(data)
    if df.empty:
      print(f"No data for year {year}, month {month}. Skipping CSV creation.")
      continue
    month_list.append(df)
    df.to_csv(f"../data/raw/daily-crime-incident-logs/{year}/daily_crime_incident_log_{year}_{month}.csv", index=False)
  ci_dfs_individual[year] = month_list
  ci_dfs_combined_yearly[year] = pd.concat(month_list, ignore_index=True)
  ci_dfs_combined_yearly[year].to_csv(f"../data/raw/daily-crime-incident-logs/daily_crime_incident_log_{year}.csv", index=False)

ci_df_all = pd.concat([df for df in ci_dfs_combined_yearly.values() if not df.empty], ignore_index=True)
ci_df_all.to_csv(f"../data/raw/daily-crime-incident-logs/daily_crime_incident_log_2019-{CURRENT_YEAR}.csv", index=False)

No data for year 2026, month 11. Skipping CSV creation.
No data for year 2026, month 12. Skipping CSV creation.


## Scraping Uniform Crime Reports

There are 3 * 4 + 1 = 13 tables to scrape on UMPD's Uniform Crime Reports website: 

- 2026 NIBRS
  - Crimes Against Persons
  - Crimes Against Property
  - Crimes Against Society
- 2025 NIBRS
  - Crimes Against Persons
  - Crimes Against Property
  - Crimes Against Society
- 2024 NIBRS
  - Crimes Against Persons
  - Crimes Against Property
  - Crimes Against Society
- 2023 NIBRS
  - Crimes Against Persons
  - Crimes Against Property
  - Crimes Against Society
- SRS (2012-2022)

In [2]:
response = requests.get(BASE_UCR_URL, headers=HEADERS)
response.raise_for_status()
soup = bs.BeautifulSoup(response.text, "html.parser")
tables = soup.find_all("table")

umpd_ucr_dfs = {}
for year in range(2023, 2027):
  umpd_ucr_dfs[f"{year}"] = {}

umpd_ucr_dfs['2012-2022'] = None

In [3]:
subtractor = 0
for i, table in enumerate(tables):
  table_name = ""
  table_position = i % 3
  
  if table_position == 0:
    table_name = "crimes_against_persons"
  elif table_position == 1:
    table_name = "crimes_against_property"
  else:
    table_name = "crimes_against_society"
  
  table_rows = table.find_all('tr')[1:] if table else []
  
  # 0, 1, 2
  # 3, 4, 5
  # 6, 7, 8
  # 9, 10, 11
  # 12
  
  
  if i >= 3 and i < 6:
    subtractor = 1
  elif i >= 6 and i < 9:
    subtractor = 2
  elif i >= 9 and i < 12:
    subtractor = 3
  elif i == 12: 
    subtractor = 4
    umpd_ucr_dfs['2012-2022'] = []
    
  cur_year = CURRENT_YEAR - subtractor
  headers = [f"{cur_year}", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", 'Sep', 'Oct', 'Nov', 'Dec', "", "Total"]
  
  if i < 12:
    
    umpd_ucr_dfs[f"{cur_year}"][table_name] = []
    
    for row in table_rows:
      
      charge = row.find("th").get_text(strip=True).strip()
      cells = row.find_all("td")
      
      umpd_ucr_dfs[f"{cur_year}"][table_name].append({
        f"{cur_year}" : charge,
        "Jan" : text(0, cells),
        "Feb": text(1, cells),
        "Mar": text(2, cells),
        "Apr": text(3, cells),
        "May": text(4, cells),
        "Jun": text(5, cells),
        "Jul": text(6, cells),
        'Aug': text(7, cells),
        'Sep': text(8, cells),
        'Oct': text(9, cells),
        'Nov': text(10, cells),
        'Dec': text(11, cells),
        "": text(12, cells),
        "Total": text(13, cells)
      })
  else: 
    
    for row in table_rows:
      charge = row.find('th').get_text(strip=True).strip()
      cells = row.find_all('td')
    
      umpd_ucr_dfs['2012-2022'].append({
        "": charge,
        "2012": text(0, cells),
        '2013': text(1, cells),
        '2014': text(2, cells),
        '2015': text(3, cells),
        '2016': text(4, cells),
        '2017': text(5, cells),
        '2018': text(6, cells),
        '2019': text(7, cells),
        '2020': text(8, cells),
        '2021': text(9, cells),
        '2022': text(10, cells)
      })
        
      
    
    

In [4]:
umpd_ucr_dfs['2012-2022']

[{'': 'Criminal Homicide',
  '2012': '0',
  '2013': '0',
  '2014': '0',
  '2015': '0',
  '2016': '0',
  '2017': '1',
  '2018': '0',
  '2019': '0',
  '2020': '0',
  '2021': '0',
  '2022': '0'},
 {'': 'Rape',
  '2012': '0',
  '2013': '0',
  '2014': '0',
  '2015': '5',
  '2016': '4',
  '2017': '3',
  '2018': '1',
  '2019': '3',
  '2020': '4',
  '2021': '1',
  '2022': '5'},
 {'': 'Robbery',
  '2012': '5',
  '2013': '3',
  '2014': '3',
  '2015': '1',
  '2016': '4',
  '2017': '3',
  '2018': '1',
  '2019': '3',
  '2020': '0',
  '2021': '0',
  '2022': '1'},
 {'': 'Aggravated Assault',
  '2012': '2',
  '2013': '6',
  '2014': '3',
  '2015': '3',
  '2016': '7',
  '2017': '7',
  '2018': '3',
  '2019': '2',
  '2020': '0',
  '2021': '2',
  '2022': '3'},
 {'': 'Burglary',
  '2012': '46',
  '2013': '51',
  '2014': '69',
  '2015': '48',
  '2016': '27',
  '2017': '20',
  '2018': '11',
  '2019': '29',
  '2020': '17',
  '2021': '17',
  '2022': '11'},
 {'': 'Larceny/Theft',
  '2012': '362',
  '2013': '405'

In [5]:
for year, val in umpd_ucr_dfs.items():
  if year == "2012-2022":
    umpd_ucr_dfs['2012-2022'] = pd.DataFrame(val)
  else: 
    for table in val: 
      table_data = umpd_ucr_dfs[year][table]
      umpd_ucr_dfs[year][table] = pd.DataFrame(table_data)

In [6]:
umpd_ucr_dfs['2025']['crimes_against_society']

,2025,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,,Total
0,Drug/Narcotic Violations,1,1,1,2,2,,2,2,5,,4,1,,21
1,Drug Equipment Violations,2,,,,1,1,2,,,,1,2,,9
2,Pornography/Obscene Material,,1,,,,,,,,,,,,1
3,Prostitution,,,,,,,,,,,,,,
4,Assisting or Promoting Prostitution,,,,,,,,,,,,,,
5,Purchasing Prostitution,,,,,,,,,,,,,,
6,Weapon Law Violations,,1,,1,1,,1,,,,,1,,5
7,Animal Cruelty,,,,,,,,,,,,,,
8,Curfew/Loitering/Vagrancy Violations,,,,,,,,,,,,,,
9,Disorderly Conduct,,2,1,1,1,1,1,,1,1,1,,,10


In [7]:
UCR_DIR = DATA_DIR / "raw" / "uniform-crime-reports"
if UCR_DIR.is_dir():
  shutil.rmtree(UCR_DIR.absolute())
  Path.mkdir(UCR_DIR)
else:
  Path.mkdir(UCR_DIR)
  
for year in umpd_ucr_dfs:
  if year == "2012-2022":
    umpd_ucr_dfs[year].to_csv(UCR_DIR / "srs_2012-2022.csv") 
  else: 
    year_path = UCR_DIR / year
    Path.mkdir(year_path)
    for table_name in umpd_ucr_dfs[year]:
      df =umpd_ucr_dfs[year][table_name]
      df.to_csv(year_path / f"{table_name}-ucr-{year}.csv")
      

## Scraping Clert Act for UMD, College Park

In [126]:
ca_cp_data = {}
response = requests.get(CLERYACT_URL, headers=HEADERS)
response.raise_for_status()
soup = bs.BeautifulSoup(response.text, "html.parser")
tables = soup.find_all('table')
ca_cp_table = tables[0]
rows = ca_cp_table.find_all("tr")[2:]
for i in range(len(rows)-2):
  row = rows[i]
  row_2024 = rows[i+1]
  row_2023 = rows[i+2]
  
  if len(row.find_all('td')) != 8:
    continue
  
  cells = row.find_all('td')
  cells_2024 = row_2024.find_all('td')
  cells_2023 = row_2023.find_all('td')
  

  
  year_data = {}
  charge = text(0, cells)
  
  list_2025 = []
  list_2024 = []
  list_2023 = []
  
  headers = 'oncampus', 'noncampus_buildings_or_property', 'public_property', 'total', '', 'oncampus_residential_facilities'
  
  for i in range(1, 8): 
    if i == 1:
      list_2024.append(text(i, cells_2024))
      list_2023.append(text(i, cells_2023))
    elif i == 7:
      list_2025.append(text(i, cells))
    else:
      list_2025.append(text(i, cells))
      list_2024.append(text(i, cells_2024))
      list_2023.append(text(i, cells_2023))
  
  year_data['2025'] = dict(zip(headers, list_2025))
  year_data['2024'] = dict(zip(headers, list_2024))
  year_data['2023'] = dict(zip(headers, list_2023))
  
  ca_cp_data[charge] = year_data


In [134]:
# Help from Codex on 2026-11-10 to address nested dictionary pandas import

clery_df = pd.concat(
  {
    charge: pd.DataFrame.from_dict(years, orient="index")
    for charge, years in ca_cp_data.items()
  },
  axis="columns",
  names = ["charge", "location"],
)

clery_df.index.name = 'year'
clery_df = clery_df.drop(columns="", level="location")
clery_df

charge   Murder & Non-negligent Manslaughter                                  \
location                            oncampus noncampus_buildings_or_property   
year                                                                           
2025                                       0                               0   
2024                                       0                               0   
2023                                       0                               0   

charge                                                          \
location public_property total oncampus_residential_facilities   
year                                                             
2025                   0     0                             (0)   
2024                   0     0                             (0)   
2023                   0     0                             (0)   

charge   Negligent Manslaughter                                  \
location               oncampus noncampus_buildings_or_property   
year                                                              
2025                          0                               0   
2024                          0                               0   
2023                          0                               0   

charge                                                          ...  \
location public_property total oncampus_residential_facilities  ...   
year                                                            ...   
2025                   0     0                             (0)  ...   
2024                   0     0                             (0)  ...   
2023                   0     0                             (0)  ...   

charge   Weapons Possession Referrals                                  \
location                     oncampus noncampus_buildings_or_property   
year                                                                    
2025                                0                               0   
2024                                0                               0   
2023                                0                               0   

charge                                                          \
location public_property total oncampus_residential_facilities   
year                                                             
2025                   0     0                             (0)   
2024                   0     0                             (0)   
2023                   0     0                             (0)   

charge   Unfounded Reports[14]                                  \
location              oncampus noncampus_buildings_or_property   
year                                                             
2025                         6                               1   
2024                         9                               0   
2023                         4                               0   

charge                                                          
location public_property total oncampus_residential_facilities  
year                                                            
2025                   0     7                             (2)  
2024                   0     9                             (1)  
2023                   0     4                             (2)  

[3 rows x 115 columns]

In [135]:
clery_df.to_csv(DATA_DIR / "raw" / "clery_act_umdcp_2023-2025.csv")

## Combining `ci_df_all` and `arrests_df_all` where they match

In [103]:
merge_results = pd.merge(arrest_df_all, ci_df_all, on="umpd_case_number", how="inner")

merge_results['umpd_case_number'].value_counts().sort_values(ascending=False)

umpd_case_number
2019-00025263    10
2021-00081637    10
2025-00088127     8
2024-00118283     6
2024-00177296     6
                 ..
2021-00035276     1
2021-00035475     1
2021-00035772     1
2021-00024023     1
2026-00289157     1
Name: count, Length: 2944, dtype: int64

In [104]:
merge_results.to_csv(f"../data/processed/merged_arrest_ci_data_2018-{CURRENT_YEAR}.csv", index=False)